# brier - M6.3: compatibility matrix for larger model families

Runs `brier check` and the integration suite (`tests/integration`, fp32) on seven more
families: LFM2-1.2B (hybrid convolution/attention), DeepSeek-R1-Distill-Qwen-1.5B (reasoning
template), Falcon3-1B, Phi-4-mini (3.8B), SmolLM3-3B (thinking mode), OLMoE-1B-7B (mixture of
experts) and Mistral-7B-v0.3. Every model is pinned to a revision; nothing is fitted or saved.

**Runtime:** A100 (40 GB is enough), about 40-50 min, mostly downloads (~50 GB). Each model's
files are deleted after its tests to keep the disk free.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRIER_COMMIT = "698ba72eb89deefd93011ad55a3acd7d902b2e17"  # M6.3 model registry
MODELS = ["lfm2", "r1distill", "falcon3", "phi4mini", "smollm3", "olmoe", "mistral7b"]

In [ ]:
!rm -rf /content/brier /content/brier_repo  # leftovers from an earlier run
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier_repo
!git -C /content/brier_repo checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier_repo[hf]" pytest hypothesis

In [ ]:
import json
import os
import shutil
import subprocess
import sys

REPO = "/content/brier_repo"
# Read the pinned model registry in a fresh process: this notebook's own Python session never
# imports brier (a session started before `pip install -e` cannot see the editable install).
REGISTRY = json.loads(
    subprocess.run(
        [
            sys.executable,
            "-c",
            "import json, sys; sys.path.insert(0, 'tests/integration');"
            "from test_hf_backend import MODELS;"
            "print(json.dumps({k: m._asdict() for k, m in MODELS.items()}))",
        ],
        capture_output=True,
        text=True,
        cwd=REPO,
        check=True,
    ).stdout
)

results = {}
for key in MODELS:
    m = REGISTRY[key]
    print(f"\n===== {key}: {m['id']} @ {m['revision'][:7]} =====", flush=True)
    check = [
        sys.executable,
        "-m",
        "brier",
        "check",
        m["id"],
        "--revision",
        m["revision"],
        "--dtype",
        "float32",
    ]
    if m["attn"]:
        check += ["--attn-implementation", m["attn"]]
    c = subprocess.run(check, capture_output=True, text=True, cwd=REPO)  # noqa: S603 - fixed args
    print(c.stdout or c.stderr[-2000:])
    env = dict(os.environ, BRIER_TEST_MODELS=key)
    t = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "-m",
            "integration",
            "-q",
            "-p",
            "no:cacheprovider",
            "--color=no",
            "tests/integration",
        ],
        capture_output=True,
        text=True,
        env=env,
        cwd=REPO,
    )
    tail = t.stdout.strip().splitlines()[-15:]
    print("\n".join(tail))
    results[key] = {
        "check_exit": c.returncode,
        "check": c.stdout,
        "pytest_exit": t.returncode,
        "pytest_tail": tail,
    }
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

In [ ]:
print("| model | brier check | integration suite |")
print("|---|---|---|")
for key, r in results.items():
    check = "OK" if r["check_exit"] == 0 else "FAILED"
    suite = r["pytest_tail"][-1] if r["pytest_tail"] else "no output"
    print(f"| {key} | {check} | {suite} |")

In [ ]:
from google.colab import files

with open("/content/m6_3_results.json", "w") as f:
    json.dump(results, f, indent=1)
files.download("/content/m6_3_results.json")